In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto"
)

model_4bit.eval()

print("4-bit model loaded successfully!")
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 3), "GB")

In [ ]:
q4_results, q4_tps, q4_memory = benchmark_model(
    model_4bit, tokenizer, prompts
)

print("4-BIT RESULTS")
display(q4_results)

print("Average TPS:", round(q4_tps, 2))
print("Peak GPU Memory:", round(q4_memory, 3), "GB")

In [ ]:
import torch
import pandas as pd

def capture_responses(model, tokenizer, prompts, model_name):
    records = []

    for i, prompt in enumerate(prompts):
        messages = [{"role": "user", "content": prompt}]

        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        ).to("cuda")

        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False
            )

        generated_ids = output[0][inputs["input_ids"].shape[1]:]
        response = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        )

        records.append({
            "Prompt_No": i + 1,
            "Prompt": prompt,
            "Response": response,
            "Model": model_name
        })

    return pd.DataFrame(records)


q4_responses = capture_responses(
    model_4bit, tokenizer, prompts, "Qwen2.5-0.5B-4bit"
)

display(q4_responses[["Prompt_No", "Response"]])

q4_responses.to_csv("q4_responses.csv", index=False)